In [9]:
# Character and word tokenization 
# Simple character level tokenizer that maps each character in an input string to an integer. Keeping it simple, using an standarization step - we lowercase all input

from typing import Any

import regex as re 

class CharTokenizer: 
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]
        
    def standarize(self, inputs): 
        return inputs.lower()
    
    def split(self, inputs): 
        return re.findall(r".", inputs)
    
    #  returns the token ID for each character inside of the list. 
    def index(self, tokens): 
        # dict.get(key, default) returns dict[key] if the key exists and default if it doesn't. 
        return[self.vocabulary.get(t, self.unk_id) for t in tokens]
    
    def __call__(self, inputs):
        inputs = self.standarize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices
         

In [10]:
import collections
# function that computes a vocabulary of tokens based on some input text. Limiting our vocabulary size to only the most common tokens in our input data. 

def compute_char_vocabulary(inputs, max_size): 
    # Class for building a new empty Counter object. 
    # Counter is a dictionary built for counting things, keys are items and values are how many times each item has been seen. 
    char_counts = collections.Counter()
    for x in inputs: 
        x = x.lower()
        tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    # The loop appens just the characters and ignores the counts. 
    for token, count in most_common: 
        vocabulary.append(token)
    # Enumerate pairs each token with its position. Dict(token, i) creates the dictionary with the tokens inside of it. 
    return dict((token, i) for i, token in enumerate(vocabulary))


In [11]:
# We can now do the same for a WORD-level tokenizer. We can use the same code as our character-level tokenizer with different splitting steps. 

class WordTokenizer: 
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]
        
    def standarize(self, inputs): 
        return inputs.lower()
    
    def split(self, inputs): 
        return re.findall(r"[\w]+|[.,!?;]", inputs)
    
    #  returns the token ID for each character inside of the list. 
    def index(self, tokens): 
        # dict.get(key, default) returns dict[key] if the key exists and default if it doesn't. 
        return[self.vocabulary.get(t, self.unk_id) for t in tokens]
    
    def __call__(self, inputs):
        inputs = self.standarize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices
         

In [12]:
# We can now substitute this new split rule into our vocabulary function. 
def compute_word_vocabulary(inputs, max_size): 
    # Class for building a new empty Counter object. 
    # Counter is a dictionary built for counting things, keys are items and values are how many times each item has been seen. 
    char_counts = collections.Counter()
    for x in inputs: 
        x = x.lower()
        tokens = re.findall(r"[\w]+|[.,!?;]", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    # The loop appens just the characters and ignores the counts. 
    for token, count in most_common: 
        vocabulary.append(token)
    # Enumerate pairs each token with its position. Dict(token, i) creates the dictionary with the tokens inside of it. 
    return dict((token, i) for i, token in enumerate(vocabulary))

In [13]:
# Lets try the tokenizers on some real world input - full text of Moby Dick by Herman Melville. First build a vocabulary for both tokenizers and then use it to tokenize some text: 

import keras

filename = keras.utils.get_file(
    origin = "https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
)
moby_dick = list(open(filename, "r", encoding= "utf-8"))

vocabulary = compute_char_vocabulary(moby_dick, max_size= 100)
print(type(vocabulary))
char_tokenizer = CharTokenizer(vocabulary)

<class 'dict'>


In [19]:
print("vocabulary lenght:", len(vocabulary))

print("vocabulary start: ", list(vocabulary.keys())[:10])

print("vocabulay end: ", list(vocabulary.keys())[-10:])

print("Line Length: ", len(char_tokenizer(
    "Call me Peter. I am old, or am I? I don't know"
)))


vocabulary lenght: 80
vocabulary start:  ['[UNK]', ' ', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h']
vocabulay end:  ['\u200e', 'ϰ', 'η', 'τ', 'ο', 'ς', "'", 'â', '%', '+']
Line Length:  46


In [20]:
vocabulary = compute_word_vocabulary(moby_dick, max_size=2000)
word_tokenizer = WordTokenizer(vocabulary)

In [21]:
print("vocabulary lenght:", len(vocabulary))

print("vocabulary start: ", list(vocabulary.keys())[:10])

print("vocabulay end: ", list(vocabulary.keys())[-10:])

print("Line Length: ", len(char_tokenizer(
    "Call me Peter. I am old, or am I? I don't know"
)))

vocabulary lenght: 2000
vocabulary start:  ['[UNK]', ',', 'the', '.', 'of', 'and', 'a', 'to', 'in', ';']
vocabulay end:  ['features', 'barbaric', 'helmsman', 'temporary', 'indispensable', 'fashion', 'lovely', 'steadily', 'fastened', 'birds']
Line Length:  46
